NAME: Mitch P. Dumdum

SECTION: BSCS - 3C

SUBJECT: Parallel and Distributed Computing

In [8]:

!pip install requests -q

import uuid
import time
import random
import requests
import threading


SUPABASE_URL = "https://mbqchufpkuzbpungfktb.supabase.co/rest/v1/"

SUPABASE_KEY = "sb_publishable_pIfpt-B8Mva6AfuDwTgHhw_wR8Xe6KK"

TABLE_NAME = "votes"


HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=representation"
}



def generate_vote(edge_id):

    return {
        "user_id": str(uuid.uuid4()),
        "edge_id": edge_id,
        "poll_id": "poll_1",
        "choice": random.choice(["A", "B", "C"]),
        "timestamp": time.time()
    }



def send_vote(vote):

    try:

        response = requests.post(
            f"{SUPABASE_URL}/{TABLE_NAME}",
            headers=HEADERS,
            json=vote,
            timeout=10
        )


        print("STATUS:", response.status_code)

        if response.text:
            print("RESPONSE:", response.text)


        if response.status_code in [200, 201]:

            print(
                f"SUCCESS | {vote['edge_id']} | "
                f"{vote['choice']}"
            )


        elif response.status_code == 409:

            print(f"DUPLICATE | {vote['edge_id']}")


        else:

            print(
                f"FAILED ({response.status_code}) "
                f"| {vote['edge_id']}"
            )

    except Exception as e:

        print(f"ERROR | {vote['edge_id']} | {e}")



def run_edge_node(edge_id, num_votes=20):

    print(f"\nSTARTING {edge_id}")

    for _ in range(num_votes):

        vote = generate_vote(edge_id)


        send_vote(vote)



        if random.random() < 0.25:

            print(f"DUPLICATE SENT FROM {edge_id}")

            send_vote(vote)


        time.sleep(random.uniform(0.5, 1.5))

    print(f"FINISHED {edge_id}")



if __name__ == "__main__":

    print("===================================")
    print("DISTRIBUTED VOTING SYSTEM STARTED")
    print("===================================")

    edges = [
        "edge-1",
        "edge-2",
        "edge-3"
    ]

    threads = []

    for edge in edges:

        t = threading.Thread(
            target=run_edge_node,
            args=(edge, 15)
        )

        threads.append(t)

        t.start()

        time.sleep(0.7)


    for t in threads:
        t.join()

    print("\n===================================")
    print("ALL EDGE NODES FINISHED")
    print("CHECK SUPABASE TABLE NOW")
    print("===================================")

DISTRIBUTED VOTING SYSTEM STARTED

STARTING edge-1

STARTING edge-2
STATUS: 201
RESPONSE: [{"id":56,"user_id":"51553161-61fe-4de4-8914-00ac09dc0387","edge_id":"edge-1","poll_id":"poll_1","choice":"B","timestamp":1779456101.07534}]
SUCCESS | edge-1 | B
DUPLICATE SENT FROM edge-1
STATUS: 201
RESPONSE: [{"id":57,"user_id":"f1d7b2a0-022e-4abe-b1b8-47b17092a0b8","edge_id":"edge-2","poll_id":"poll_1","choice":"C","timestamp":1779456101.77734}]
SUCCESS | edge-2 | C
DUPLICATE SENT FROM edge-2

STARTING edge-3
STATUS: 201
RESPONSE: [{"id":58,"user_id":"f1d7b2a0-022e-4abe-b1b8-47b17092a0b8","edge_id":"edge-2","poll_id":"poll_1","choice":"C","timestamp":1779456101.77734}]
SUCCESS | edge-2 | C
STATUS: 201
RESPONSE: [{"id":59,"user_id":"51553161-61fe-4de4-8914-00ac09dc0387","edge_id":"edge-1","poll_id":"poll_1","choice":"B","timestamp":1779456101.07534}]
SUCCESS | edge-1 | B
STATUS: 201
RESPONSE: [{"id":60,"user_id":"7f261a3d-6b5e-422d-a2cb-63dfb9a99eae","edge_id":"edge-3","poll_id":"poll_1","choic